# Validation: aFRR Hourly Activation Price Methodologies

This notebook compares three hourly price methodologies for aFRR activation (POS/NEG):

1. Official Average (`GERMANY_AVERAGE_ENERGY_PRICE_[EUR/MWh]`)
2. Official Marginal (`GERMANY_MARGINAL_ENERGY_PRICE_[EUR/MWh]`)
3. Anonymous Bids (`afrr_bid_vwap_activation_price` derived from tick-level bids)

All merges are performed on `timestamp_utc`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import io
import re
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from energy_trading.visualization.style import apply_geo_style
    apply_geo_style()
except Exception:
    plt.style.use('default')

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

PRIMARY = '#226E9C'
COL_AVG = '#d9b98d'
COL_MARG = '#7C1D6F'
COL_BID = '#089099'
COL_VOL = '#999999'
PICASSO_UTC = pd.Timestamp('2022-06-22 00:00:00+00:00')

ROOT = Path('..')
MARGINAL_DIR = ROOT / 'data/raw/marginal_prices'
BIDS_DIR = ROOT / 'data/raw/bids'
NETZ_PATH = ROOT / 'data/raw/netztransparenz.parquet'
VOLUME_15_PATH = ROOT / 'data/raw/volumes_15min.parquet'
REGEL_PATH = ROOT / 'data/raw/regelleistung.parquet'


## 1) Helper functions (parsing + aggregation)

Notes:
- `PRODUCT` may appear as `POS_001..POS_096` (15-min index) or `POS_04_08` (hour block).
- For block products, rows are expanded to quarter-hours uniformly inside the block.
- If no dedicated 15-min volume parquet exists, hourly netztransparenz MWh are distributed equally to 15-min (fallback).


In [ ]:
def _normalize_col_name(name: str) -> str:
    return ''.join(ch for ch in str(name).lower() if ch.isalnum())


def _parse_date_series(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, dayfirst=True, errors='coerce')
    serial = pd.to_numeric(series, errors='coerce')
    use_serial = serial.notna() & (parsed.isna() | (parsed.dt.year < 1995))
    if use_serial.any():
        parsed.loc[use_serial] = pd.to_datetime(
            serial.loc[use_serial], unit='D', origin='1899-12-30', errors='coerce'
        )
    return parsed


def _iter_excel_frames(path: Path):
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path, 'r') as zf:
            for name in zf.namelist():
                if name.lower().endswith(('.xlsx', '.xls')):
                    with zf.open(name) as f:
                        xls = pd.ExcelFile(io.BytesIO(f.read()), engine='openpyxl')
                        for sheet in xls.sheet_names:
                            yield f'{path.name}/{name}:{sheet}', xls.parse(sheet_name=sheet)
    else:
        xls = pd.ExcelFile(path, engine='openpyxl')
        for sheet in xls.sheet_names:
            yield f'{path.name}:{sheet}', xls.parse(sheet_name=sheet)


def _expand_product_to_timestamps(date_local: pd.Timestamp, product: str) -> tuple[str, list[pd.Timestamp]]:
    p = str(product).upper().strip()
    m_qh = re.match(r'^(POS|NEG)_(\d{3})$', p)
    if m_qh:
        direction, q = m_qh.group(1), int(m_qh.group(2))
        if not (1 <= q <= 96):
            return direction, []
        base = pd.Timestamp(date_local).floor('D').tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward')
        ts = base + pd.Timedelta(minutes=(q - 1) * 15)
        return direction, [ts.tz_convert('UTC')]

    m_blk = re.match(r'^(POS|NEG)_(\d{2})_(\d{2})$', p)
    if m_blk:
        direction, sh, eh = m_blk.group(1), int(m_blk.group(2)), int(m_blk.group(3))
        base = pd.Timestamp(date_local).floor('D').tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward')
        hours = list(range(sh, eh if eh > sh else eh + 24))
        out = []
        for h in hours:
            for m in (0, 15, 30, 45):
                out.append((base + pd.Timedelta(hours=h, minutes=m)).tz_convert('UTC'))
        return direction, out

    return 'UNK', []


def _to_hourly_vwap(df_15: pd.DataFrame, price_col: str, vol_col: str, out_col: str) -> pd.DataFrame:
    x = df_15.copy()
    x['hour'] = x['timestamp_utc'].dt.floor('1h')
    x['pv'] = x[price_col] * x[vol_col]
    g = (
        x.groupby(['hour', 'direction'], as_index=False)
         .agg(num=('pv', 'sum'), den=(vol_col, 'sum'))
    )
    g[out_col] = np.where(g['den'] != 0, g['num'] / g['den'], np.nan)
    return g.rename(columns={'hour': 'timestamp_utc'})[['timestamp_utc', 'direction', out_col, 'den']]


## 2) Load official 15-min prices (Average + Marginal)

In [ ]:
def load_official_prices_15m(marginal_dir: Path) -> pd.DataFrame:
    files = sorted(marginal_dir.glob('RESULT_OVERVIEW_ENERGY_MARKET_aFRR_*.xlsx'))
    rows = []
    for path in files:
        for src, df in _iter_excel_frames(path):
            df.columns = [str(c).strip() for c in df.columns]
            cmap = {_normalize_col_name(c): c for c in df.columns}

            date_col = cmap.get('deliverydate')
            prod_col = cmap.get('product')
            avg_col = cmap.get('germanyaverageenergypriceeurmwh')
            marg_col = cmap.get('germanymarginalenergypriceeurmwh')

            if not (date_col and prod_col and avg_col and marg_col):
                continue

            tmp = df[[date_col, prod_col, avg_col, marg_col]].copy()
            tmp[date_col] = _parse_date_series(tmp[date_col])
            tmp[avg_col] = pd.to_numeric(tmp[avg_col], errors='coerce')
            tmp[marg_col] = pd.to_numeric(tmp[marg_col], errors='coerce')
            tmp = tmp.dropna(subset=[date_col, prod_col])

            for r in tmp.itertuples(index=False):
                d_local = getattr(r, date_col)
                product = getattr(r, prod_col)
                p_avg = getattr(r, avg_col)
                p_marg = getattr(r, marg_col)
                direction, ts_list = _expand_product_to_timestamps(d_local, product)
                if direction not in ('POS', 'NEG') or not ts_list:
                    continue
                for ts in ts_list:
                    rows.append((ts, direction, p_avg, p_marg))

    out = pd.DataFrame(rows, columns=['timestamp_utc', 'direction', 'price_avg_15m', 'price_marg_15m'])
    if out.empty:
        return out
    out = (
        out.groupby(['timestamp_utc', 'direction'], as_index=False)
           .agg(price_avg_15m=('price_avg_15m', 'mean'), price_marg_15m=('price_marg_15m', 'mean'))
           .sort_values(['timestamp_utc', 'direction'])
    )
    return out


official_15 = load_official_prices_15m(MARGINAL_DIR)
print('official_15 rows:', len(official_15))
official_15.head(3)


## 3) Load anonymous bids (15-min VWAP from tick-level bids)

In [ ]:
def _bid_usecol(name: str) -> bool:
    n = _normalize_col_name(name)
    keys = (
        'deliverydate', 'product', 'typeofreserves', 'reservetype', 'country',
        'allocatedcapacity', 'capacitymw', 'energyprice', 'paymentdirection'
    )
    return any(k in n for k in keys)


def load_bid_vwap_15m(bids_dir: Path) -> pd.DataFrame:
    files = sorted(
        p for p in bids_dir.rglob('RESULT_LIST_ANONYM_ENERGY_MARKET_aFRR_*.xlsx*')
        if p.is_file() and not p.name.startswith('~$')
    )
    rows = []
    for path in files:
        for src, df in _iter_excel_frames(path):
            df.columns = [str(c).strip() for c in df.columns]
            # prune to relevant columns if present
            keep = [c for c in df.columns if _bid_usecol(c)]
            if keep:
                df = df[keep].copy()

            cmap = {_normalize_col_name(c): c for c in df.columns}
            date_col = cmap.get('deliverydate')
            prod_col = cmap.get('product')
            alloc_col = cmap.get('allocatedcapacitymw') or cmap.get('allocatedcapacity') or cmap.get('capacitymw')
            price_col = cmap.get('energypriceeurmwh') or cmap.get('energyprice')
            pay_col = cmap.get('energypricepaymentdirection')

            if not (date_col and prod_col and alloc_col and price_col):
                continue

            t = df[[date_col, prod_col, alloc_col, price_col] + ([pay_col] if pay_col else [])].copy()
            t[date_col] = _parse_date_series(t[date_col])
            t[alloc_col] = pd.to_numeric(t[alloc_col], errors='coerce')
            t[price_col] = pd.to_numeric(t[price_col], errors='coerce')
            t = t.dropna(subset=[date_col, prod_col, alloc_col, price_col])

            if 'typeofreserves' in cmap:
                rc = cmap['typeofreserves']
                t = t[t[rc].astype(str).str.upper().str.contains('AFRR', na=False)]
            if 'country' in cmap:
                cc = cmap['country']
                t = t[t[cc].astype(str).str.upper().eq('DE')]

            for r in t.itertuples(index=False):
                d_local = getattr(r, date_col)
                product = getattr(r, prod_col)
                alloc = float(getattr(r, alloc_col))
                price = float(getattr(r, price_col))
                pay = str(getattr(r, pay_col)).upper() if pay_col else ''

                direction, ts_list = _expand_product_to_timestamps(d_local, product)
                if direction not in ('POS', 'NEG') or not ts_list:
                    continue

                pay_sign = -1.0 if 'PROVIDER_TO_GRID' in pay else 1.0
                dir_sign = -1.0 if direction == 'NEG' else 1.0
                price_signed = price * pay_sign * dir_sign

                for ts in ts_list:
                    rows.append((ts, direction, price_signed, alloc))

    b = pd.DataFrame(rows, columns=['timestamp_utc', 'direction', 'price_signed', 'alloc_mw'])
    if b.empty:
        return b
    b['pv'] = b['price_signed'] * b['alloc_mw']
    out = (
        b.groupby(['timestamp_utc', 'direction'], as_index=False)
         .agg(num=('pv', 'sum'), den=('alloc_mw', 'sum'))
    )
    out['bid_vwap_15m'] = np.where(out['den'] != 0, out['num'] / out['den'], np.nan)
    return out[['timestamp_utc', 'direction', 'bid_vwap_15m']].sort_values(['timestamp_utc', 'direction'])


bid_15 = load_bid_vwap_15m(BIDS_DIR)
print('bid_15 rows:', len(bid_15))
bid_15.head(3)


## 4) Load 15-min activated volumes (netztransparenz)

In [ ]:
def load_activated_volume_15m(netz_path: Path, preferred_15_path: Path) -> pd.DataFrame:
    if preferred_15_path.exists():
        v = pd.read_parquet(preferred_15_path)
        v['timestamp_utc'] = pd.to_datetime(v['timestamp_utc'], utc=True)
        cols = set(v.columns)

        pos_col = next((c for c in ['afrr_activated_mwh_pos', 'activated_mwh_pos', 'vol_pos_mwh'] if c in cols), None)
        neg_col = next((c for c in ['afrr_activated_mwh_neg', 'activated_mwh_neg', 'vol_neg_mwh'] if c in cols), None)
        if not (pos_col and neg_col):
            raise KeyError('Could not find POS/NEG 15-min volume columns in volumes_15min.parquet')

        long = pd.concat([
            v[['timestamp_utc', pos_col]].rename(columns={pos_col: 'vol_mwh'}).assign(direction='POS'),
            v[['timestamp_utc', neg_col]].rename(columns={neg_col: 'vol_mwh'}).assign(direction='NEG'),
        ], ignore_index=True)
        long['vol_mwh'] = pd.to_numeric(long['vol_mwh'], errors='coerce')
        return long.dropna(subset=['timestamp_utc']).sort_values(['timestamp_utc', 'direction'])

    # Fallback: upsample hourly netztransparenz MWh equally to 15-minute slices.
    n = pd.read_parquet(netz_path)
    n['timestamp_utc'] = pd.to_datetime(n['timestamp_utc'], utc=True)
    need = ['timestamp_utc', 'afrr_activated_mwh_pos', 'afrr_activated_mwh_neg']
    miss = [c for c in need if c not in n.columns]
    if miss:
        raise KeyError(f'Missing required columns in netztransparenz.parquet: {miss}')

    base = n[need].copy()
    base['afrr_activated_mwh_pos'] = pd.to_numeric(base['afrr_activated_mwh_pos'], errors='coerce').fillna(0.0)
    base['afrr_activated_mwh_neg'] = pd.to_numeric(base['afrr_activated_mwh_neg'], errors='coerce').fillna(0.0)

    parts = []
    for m in (0, 15, 30, 45):
        t = base.copy()
        t['timestamp_utc'] = t['timestamp_utc'] + pd.Timedelta(minutes=m)
        t['afrr_activated_mwh_pos'] = t['afrr_activated_mwh_pos'] / 4.0
        t['afrr_activated_mwh_neg'] = t['afrr_activated_mwh_neg'] / 4.0
        parts.append(t)

    qh = pd.concat(parts, ignore_index=True)
    long = pd.concat([
        qh[['timestamp_utc', 'afrr_activated_mwh_pos']].rename(columns={'afrr_activated_mwh_pos': 'vol_mwh'}).assign(direction='POS'),
        qh[['timestamp_utc', 'afrr_activated_mwh_neg']].rename(columns={'afrr_activated_mwh_neg': 'vol_mwh'}).assign(direction='NEG'),
    ], ignore_index=True)
    long['vol_mwh'] = pd.to_numeric(long['vol_mwh'], errors='coerce').fillna(0.0)

    print('NOTE: volumes_15min.parquet not found; used fallback from hourly netztransparenz (equal split /4).')
    return long.sort_values(['timestamp_utc', 'direction'])


vol_15 = load_activated_volume_15m(NETZ_PATH, VOLUME_15_PATH)
print('vol_15 rows:', len(vol_15))
vol_15.head(3)


## 5) Hourly VWAP aggregation + settlement truth

In [ ]:
official_w = official_15.merge(vol_15, on=['timestamp_utc', 'direction'], how='inner')
bid_w = bid_15.merge(vol_15, on=['timestamp_utc', 'direction'], how='inner')

h_avg = _to_hourly_vwap(official_w, 'price_avg_15m', 'vol_mwh', 'p_official_avg_h').drop(columns=['den'])
h_marg = _to_hourly_vwap(official_w, 'price_marg_15m', 'vol_mwh', 'p_official_marg_h').drop(columns=['den'])
h_bid = _to_hourly_vwap(bid_w, 'bid_vwap_15m', 'vol_mwh', 'p_bid_h').drop(columns=['den'])

vol_h = (
    vol_15.assign(hour=lambda x: x['timestamp_utc'].dt.floor('1h'))
          .groupby(['hour', 'direction'], as_index=False)
          .agg(activated_mwh_h=('vol_mwh', 'sum'))
          .rename(columns={'hour': 'timestamp_utc'})
)

cmp = (
    h_avg.merge(h_marg, on=['timestamp_utc', 'direction'], how='outer')
         .merge(h_bid, on=['timestamp_utc', 'direction'], how='outer')
         .merge(vol_h, on=['timestamp_utc', 'direction'], how='left')
         .sort_values(['timestamp_utc', 'direction'])
)

cmp['p_settlement_truth'] = np.where(
    cmp['timestamp_utc'] < PICASSO_UTC,
    cmp['p_official_avg_h'],
    cmp['p_official_marg_h'],
)
cmp['err_bid_vs_truth'] = cmp['p_bid_h'] - cmp['p_settlement_truth']
cmp['abs_err_bid_vs_truth'] = cmp['err_bid_vs_truth'].abs()
cmp['regime'] = np.where(cmp['timestamp_utc'] < PICASSO_UTC, 'pre_picasso', 'post_picasso')

cmp.head()


## 6) Error diagnostics (Anonymous Bids vs Settlement Truth)

In [ ]:
diag = (
    cmp.groupby(['direction', 'regime'], as_index=False)
       .agg(
           rows=('err_bid_vs_truth', lambda s: s.notna().sum()),
           mae=('abs_err_bid_vs_truth', 'mean'),
           bias=('err_bid_vs_truth', 'mean'),
           rmse=('err_bid_vs_truth', lambda s: np.sqrt(np.nanmean(np.square(s))))
       )
)

diag


## 7) Visualization: prices + volume (dual axis)

In [ ]:
def plot_dual_axis(df: pd.DataFrame, direction: str, start: str | None = None, end: str | None = None):
    x = df[df['direction'] == direction].copy()
    if start:
        x = x[x['timestamp_utc'] >= pd.Timestamp(start, tz='UTC')]
    if end:
        x = x[x['timestamp_utc'] <= pd.Timestamp(end, tz='UTC')]

    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()

    ax1.plot(x['timestamp_utc'], x['p_official_avg_h'], color=COL_AVG, linewidth=1.8, label='Official Avg (hourly VWAP)')
    ax1.plot(x['timestamp_utc'], x['p_official_marg_h'], color=COL_MARG, linewidth=1.8, label='Official Marginal (hourly VWAP)')
    ax1.plot(x['timestamp_utc'], x['p_bid_h'], color=PRIMARY, linewidth=2.2, label='Anonymous Bids (hourly VWAP)')

    ax2.bar(x['timestamp_utc'], x['activated_mwh_h'], width=0.03, color=COL_VOL, alpha=0.25, label='Activated volume (MWh)')

    ax1.axvline(PICASSO_UTC, color='black', linestyle='--', linewidth=1.2)
    ax1.text(PICASSO_UTC, ax1.get_ylim()[1] * 0.95, 'PICASSO go-live
2022-06-22', ha='left', va='top', fontsize=9)

    ax1.set_title(f'aFRR {direction}: Hourly prices vs activated volume')
    ax1.set_xlabel('timestamp_utc')
    ax1.set_ylabel('Price [EUR/MWh]')
    ax2.set_ylabel('Activated volume [MWh]')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    plt.show()


plot_dual_axis(cmp, 'POS')
plot_dual_axis(cmp, 'NEG')


## 8) Visualization: error over time + PICASSO effect

In [ ]:
for direction in ['POS', 'NEG']:
    x = cmp[cmp['direction'] == direction].copy()

    fig, ax = plt.subplots(figsize=(14, 4.8))
    ax.plot(x['timestamp_utc'], x['err_bid_vs_truth'], color=PRIMARY, linewidth=1.2, label='Error: bids - settlement truth')
    ax.axhline(0.0, color='#333333', linewidth=1.0)
    ax.axvline(PICASSO_UTC, color='black', linestyle='--', linewidth=1.2)
    ax.axvspan(PICASSO_UTC, x['timestamp_utc'].max(), color='#f4f4f4', alpha=0.4, label='Post-PICASSO')

    ax.set_title(f'{direction}: mismatch between local German bid VWAP and settlement truth')
    ax.set_xlabel('timestamp_utc')
    ax.set_ylabel('Residual [EUR/MWh]')
    ax.legend(loc='upper left')
    plt.show()


## 9) Interpretation checklist

- `p_settlement_truth` is regime-dependent:
  - `< 2022-06-22`: official average price
  - `>= 2022-06-22`: official marginal price
- Increasing post-2022 residuals are consistent with the PICASSO coupling effect:
  German anonymous bids represent local offers, while settlement is affected by cross-border activation.
- Validate timezone consistency by keeping all processed timestamps in `timestamp_utc`.
